In [12]:
from bs4 import BeautifulSoup
import numpy as np
import re, os, email, json, numbers, pickle, tqdm, time
from sympy import symbols, sympify
import os, pickle, json
from groq import Groq

api_root = os.path.expanduser('~/.api_keys')
api_key = open(os.path.join(api_root, 'groq')).read().strip()
client = Groq(api_key=api_key)
data_root = '/home/jhyang/WORKSPACES/DATA/ICSD_articles'
x = symbols('x')

with open('/home/jhyang/WORKSPACES/DATA/ICSD_articles/parsed_documents.pkl','rb') as f:
    docs = pickle.load(f)
dois = list(docs.keys())

def to_chunk(text, chunk=100):
    sentences = []
    for sentence in text.split('\n'):
        if len(sentence) == 0:
            continue
        s = []
        for token in re.split(r'( )+', sentence):
            if (len(token) == 1) and (len(token.strip()) == 0):
                continue
            s.append(token)
            if len(' '.join(s)) > chunk:
                sentences.append(' '.join(s))
                s = []
    return sentences

def highlight_text(sentences, keys):
    output = []
    for sentence in sentences:
        for k in keys:
            if k in sentence:
                sentence = sentence.replace(k, f'\x1b[43m\x1b[30m{k}\x1b[0m')
        output.append(sentence)
    return output

def print_chunk(sentences, f=None):
    for sentence in sentences:
        if f is None:
            print(sentence)
        else:
            f.write(sentence + '\n')

# Parsing document

## read

### read source

In [2]:
paths = []
soups = []
htmls = []
for pub in os.listdir(os.path.join(data_root, 'src')):
    path_pub = os.path.join(data_root, 'src', pub)
    for n in os.listdir(path_pub):
        path = os.path.join(path_pub, n)
        if os.path.isfile(path):
            jn = ''
            paths.append([pub, jn, path])
        else:
            jn = n
            for fn in os.listdir(path):
                paths.append([pub, jn, os.path.join(path, fn)])

for pub, jn, path_fn in paths:
    with open(path_fn, 'r', encoding='utf-8') as f:
        if path_fn.endswith('.mhtml'):
            msg = email.message_from_file(f)
            for part in msg.walk():
                if part.get_content_type() == 'text/html':
                    break
            html = part.get_payload(decode=True).decode('utf-8')
        else:
            html = f.read()
        soup = BeautifulSoup(html, 'html.parser')
        soups.append(soup)
        htmls.append(html)

### DOI 

In [3]:
doi_full_pattern = re.compile(r'(?:https?://doi\.org/|doi:)\s*(10\.\d{4,9}/[-._;()/:A-Z0-9]+)', re.IGNORECASE)

def parse_doi(text):
    match = doi_full_pattern.search(text)
    if match:
        return match.group(1)
    else:
        return text

In [4]:
dois = []
for i, (soup, (pub, jn, html)) in enumerate(zip(soups, paths)):
    _dois = []
    for meta in soup.find_all('meta'):
        if meta.get('name') is None:
            continue
        if 'doi' in meta.get('name'):
            _dois.append(parse_doi(meta.get('content')))
    _dois = np.unique(_dois)
    if len(_dois) == 1:
        dois.append(_dois[0])
    else:
        print(i, pub, jn, html)
        [print(doi) for doi in _dois]
        # check
    

### synthesis paragraph

In [7]:
count = []
hist = {'abs':[], 'intro':[], 'conc':[], 'ref':[], 'ack':[]}
for soup in soups:
    headers = soup.find_all('h1 h2 h3 h4 h5 h6'.split())
    c = [False, False, False, False, False]
    for i, header in enumerate(headers):

        text = header.get_text().lower()
        if 'abstract' in text:
            c[0] = True
            hist['abs'].append(text)
        if 'intro' in text:
            c[1] = True
            hist['intro'].append(text)
        if 'conclu' in text or 'outlook' in text:
            c[2] = True
            hist['conc'].append(text)
        if 'reference' in text:
            c[3] = True
            hist['ref'].append(text)
        if 'acknowledg' in text:
            c[4] = True
            hist['ack'].append(text)
    count.append(c)
count = np.array(count)

In [68]:
def get_main_text(soup):
    doc = {}
    collect = False
    removed = []
    for i, tag in enumerate(soup.find_all('h1 h2 h3 h4 h5 h6 p div span'.split())):
        if tag.find('h1 h2 h3 h4 h5 h6 p div'.split()) is not None:
            continue
        text = tag.get_text().strip()
        if len(text) == 0:
            continue
        if (tag.name == 'span') and (len(text) < 100):
            removed.append((i, text))
            continue
        if tag.name.startswith('h'):
            if any([x in text.lower() for x in ['intro','abstract','keyword']]):
                collect = True
            if any([text.lower().startswith(x) for x in ['refere','acknowledg','credit','appendix','conflict']]):
                break
            if any([x in text.lower() for x in ['intro','keyword','reference','license','author','supporting','interest','supplement']]) or any([text.lower().startswith(x) for x in ['fig','table']]):
                header = None
            elif collect and ((i, text) not in doc.keys()):
                header = (i, text)
                doc[header] = []
            if (not collect) or (header is None):
                removed.append((i, text))
            continue
#        print(collect, header, text)
        if not collect: 
            continue
        if header is None:
            continue
        doc[header].append((i, text))
    return {k:v for k,v in doc.items() if len(v) != 0}, removed

def print_document(doc, chunk=200):
#    blacklist = ['intro','abstract','conclu','acknowledg','reference','license','author','supporting','table','figure','keyword','interest']
    whitelist = ['synthe','prep','heat','anneal','pressed','cool']
    for (i, header), paragraph in doc.items():
#        if any([x in header.lower() for x in blacklist]):
#            continue
#        if len(paragraph) == 0:
#            continue
        print('='*chunk)
        print('{:>7s}\t{}'.format(f'({i})', header))
        print('-'*chunk)
        for (j, sentence) in paragraph:
            sentence = sentence.replace('\n',' ')
            for k in range(0, len(sentence), chunk):
                s = sentence[k:k+chunk]
                for x in whitelist:
                    s = s.replace(x,f'\x1b[43m\x1b[30m{x}\x1b[0m')
                if k == 0:
                    print('{:6d} \t{}'.format(j, s))
                else:
                    print('       \t{}'.format(s))
    for i in range(5): print()
    print('==     E   N   D     =='.center(chunk))
    for i in range(5): print()

def update_documents(documents, doi, soup, idxs):
    if isinstance(idxs, numbers.Integral):
        _idxs = [idxs]
    elif isinstance(idxs, str):
        _idxs = [int(l) for l in idxs.replace(',',' ').strip().split()]
    elif isinstance(idxs, (list, tuple)):
        _idxs = [int(l) for l in idxs]
    elements = soup.find_all('h1 h2 h3 h4 h5 h6 p div'.split())
    documents[doi] = [elements[idx].get_text() for idx in sorted(_idxs)]
    return documents

def print_soup(tag, level=0):
    print('  ' * level, f'<{tag.name}>', tag.text.strip().replace('\n','<br>'))
    for child in tag.children:
        if child.name:
            print_soup(child, level + 1)

In [ ]:
#dois[210]
doc = get_main_text(soups[201])
print_document(doc[0], 180)    
for (i, s) in doc[1]:
    print(f'{i}\t{s}')

In [65]:
#eles = soups[201].find_all('h1 h2 h3 h4 h5 h6 p div span'.split())
eles = soups[201].find_all('h1 h2 h3 h4 h5 h6 p div span'.split())
#parsed_eles = [ele for ele in eles if ele.find('h1 h2 h3 h4 h5 h6 p div'.split()) is None]
for i, ele in enumerate(eles):
    if ele.find('h1 h2 h3 h4 h5 h6 p div'.split()) is not None:
        continue
    n = len(ele.text.replace('\n','').strip())
    if (ele.name == 'span') and (n < 100):
        print('skip',ele.text)
        print('- '*50)
        continue

#    parents = ele.find_parents('h1 h2 h3 h4 h5 h6 p div'.split())[:2]
#    for p in parents:
#        print(p.name, p.text)
#    print(i, ele)
#    print('-'*100)
    if i > 120: break

skip 
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
skip View PDF Version
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
skip Previous Article
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
skip Next Article
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
skip J. Mater. Chem. A
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
skip 
        
          
            Stefan 
            Strangmüller
          
        
      
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
skip 
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
skip a
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

In [70]:
docs = {}
for i, (doi, soup) in enumerate(zip(dois, soups)):
    doc, _ = get_main_text(soup)
    docs[doi] = doc
    if len(doc) == 0:
        print(i, doi)

with open(os.path.join(data_root, 'parsed_documents_v2.pkl'),'wb') as f:
    pickle.dump(docs, f)

185 10.5194/ejm-35-373-2023
226 10.1107/S2414314621007355
227 10.1107/S2056989021000633
229 10.1107/S2414314621009883


In [271]:
i = 185
doi = dois[i]
doc, _ = get_main_text(soups[i])
for k, v in doc.items():
    print(k, len(v))
print('=')
for k, v in docs_[doi].items():
    print(k, len(v))

=


In [273]:
soup = soups[185]#print_soup(soups[181])
print_context = True
for i, tag in enumerate(soup.find_all('h1 h2 h3 h4 h5 h6'.split())):
#    if tag.find('h1 h2 h3 h4 h5 h6 p div'.split()) is not None:
#        continue
#    if tag.name != 'p':
#        if 'abstract' in tag.get_text().lower():
#            print_context = True
#        elif 'acknowledge' in tag.get_text().lower():
#            print_context = False
    if print_context:
        print(i, tag.name, tag.text.strip().replace('\n','<b>'))

0 h1 Article
1 h1 hits for
2 h1 Network problems
3 h1 Server timeout
4 h1 Empty search term
5 h1 Too many requests
6 h1 Fe-bearing vanadium dioxide–paramontroseite:  structural details and high-temperature transformation
7 h3 Nadia Curetti
8 h3 Alessandro Pavese
9 h2 2.1 Sample
10 h2 2.2 Chemical maps and analyses
11 h2 2.3 Single-crystal X-ray diffraction
12 h2 2.4 In situ high-temperature X-ray powder diffraction
13 h2 2.5 Raman spectroscopy on single crystals
14 h2 3.1 Single-crystal X-ray diffraction
15 h2 3.2 In situ high-temperature X-ray powder diffraction
16 h2 3.3 Raman spectroscopy on natural and heated crystals


## find synthesis paragraphs

In [105]:
with open('../dump/documents_synthesis_only.json','r') as f:
    documents = json.load(f)
processed_dois = [dois.index(k) for k in documents.keys()]
m = np.ones_like(dois, dtype=bool)
m[processed_dois] = False
m[np.max(processed_dois):] = False
len(documents), np.max(processed_dois), np.where(m)[0]

(109, 117, array([ 5, 30, 33, 48, 56, 62, 69, 90, 94]))

In [ ]:
i = 117
doc, removed = get_main_text(soups[i])
print(f'https://doi.org/{dois[i]}')
print_document(doc, chunk=190)

In [301]:
idxs = '84 101 162'
documents = update_documents(documents, dois[i], soups[i], idxs)

with open('../dump/documents_synthesis_only.json','w') as f:
    json.dump(documents, f)

In [ ]:
eles = soups[i].find_all('h1 h2 h3 h4 h5 h6 p div'.split())
print(eles[125])

In [ ]:
chunk = 150
for doi, para in documents.items():
    print(doi)
    for sentence in para:
        for i in range(0, len(sentence), chunk):
            print(f'\t{sentence[i:i+chunk]}')

In [ ]:
html = htmls[0]
#html = html.replace('>','>\n')
while '\n\n' in html:
    html = html.replace('\n\n','\n')
lines = [l for l in html.split('\n') if len(l.strip()) != 0]
for i,l in enumerate(lines):
    if 'intro' in l.lower():
        print('='*50)
    print(f'{i}\t{l}')


# LLM APIs

## groq api

In [7]:
def chat(msg, model='mixtral-8x7b-32768'):
    if isinstance(msg, str):
        messages = [{'role':'user', 'content':msg}]
    elif isinstance(msg, (list, tuple)):
        messages = [{'role':'user', 'content':_msg} for _msg in msg]
    else:
        raise ValueError('Invalid input for chat', type(msg))
    response = client.chat.completions.create(messages=messages, model=model)
    return response

def print_chat(choices, maxlen=95):
    for choice in choices:
        text = choice.message.content.split('\n')
        code = False
        for _text in text:
            if _text.startswith('```'):
                code = not code
            if code:
                print(_text)
            else:
                c = 0
                for token in _text.split():
                    c += len(token) + 1
                    if c > maxlen:
                        print()
#                        print(c, token)
                        c = len(token) + 1
                    print(token, end=' ')
                print()

In [50]:
message = chat(
   [
       'I will give you chunk of html documents that consisting single article. this includes various useless informations. I need text part that starts from intro or abstract to conclusion. all the other part such as title, metadata, supporting info, acknowledgements are not needed. give me the main part of the article in text format'
    ] + [
        htmls[0][i:i+5000] for i in range(0, len(htmls[0]), 5000)
   ],
)
print_chat(message.choices)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `mixtral-8x7b-32768` in organization `org_01j55mv9bdfxfvmjcgptqqrrb2` on tokens per minute (TPM): Limit 5000, Requested 168062, please reduce your message size and try again. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## langchain-groq

In [8]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(
    model='llama3-70b-8192',
    temperature=0.0,
    max_tokens=None,
    api_key=api_key
)

### summary of document

In [10]:
prompt1 = ChatPromptTemplate.from_messages([
    ("system", open('../dump/prompts/langchain_groq_p1.txt').read()),
    ("user","<PREVIOUS_SUMMARY>\n{summary}"),
    ("user","<DOCUMENT_CHUNK>\n{chunk}"),
])

prompt2 = ChatPromptTemplate.from_messages([
    ("system", open('../dump/prompts/langchain_groq_p2.txt').read()),
    ("user","SUMMARIES:\n{summary}"),
])

chain1 = prompt1 | llm
chain2 = prompt2 | llm

In [ ]:
max_chunk_size = 8000
for i, doi in enumerate(dois):
    if os.path.isfile('../dump/synthesis/{}.json'.format(doi.replace('/','%'))):
        continue
    print(i, '\t', doi)
    responses = []
    chunk = ''
    summary = ''
    for (_, header), section in docs[doi].items():
        chunk += f'SECTION: {header}\n'
        for (_, paragraph) in section:
            chunk += paragraph + '\n'
            if len(chunk) > max_chunk_size:
                while True:
                    try:
                        response =  chain1.invoke({'summary':summary, 'chunk':chunk})
                        break
                    except:
                        time.sleep(60)
                summary = response.content
                responses.append(summary)
                chunk = f'SECTION: {header}\n'

    if len(chunk) != 0:
        while True:
            try:
                response =  chain1.invoke({'summary':summary, 'chunk':chunk})
                break
            except:
                time.sleep(60)
        responses.append(response.content)

    summary = ''
    if len(responses) == 0:
        with open('../dump/synthesis/{}.json'.format(doi.replace('/','%')),'w') as f:
            json.dump([], f, indent=4)
        continue
    for i, response in enumerate(responses):
        summary += f'CHUNK {i}:\n{response}\n\n'
    i = 0
    while i < 10:
        try:
            output = chain2.invoke({'summary':summary})
        except:
            time.sleep(60)
            continue
        try:
            i, j = output.content.index('<SOL>'), output.content.index('<EOL>')
            output_data = eval(output.content[i+5:j])
            output_data.append({'responses':responses})
            with open('../dump/synthesis/{}.json'.format(doi.replace('/','%')),'w') as f:
                json.dump(output_data, f, indent=4)
            break
        except:
            with open('../dump/synthesis/{}.json'.format(doi.replace('/','%')),'w') as f:
                json.dump([], f, indent=4)
            i += 1
            continue

In [11]:
for doi in tqdm.tqdm(dois):
    if os.path.isfile('../dump/synthesis/{}.json'.format(doi.replace('/','%'))):
        continue
        max_chunk_size = 8000
    responses = []
    chunk = ''
    summary = ''
    for (_, header), section in docs[doi].items():
        chunk += f'SECTION: {header}\n'
        for (_, paragraph) in section:
            chunk += paragraph + '\n'
            if len(chunk) > max_chunk_size:
                while True:
                    try:
                        response =  chain1.invoke({'summary':summary, 'chunk':chunk})
                        break
                    except:
                        time.sleep(60)
                summary = response.content
                responses.append(summary)
                chunk = f'SECTION: {header}\n'

    if len(chunk) != 0:
        while True:
            try:
                response =  chain1.invoke({'summary':summary, 'chunk':chunk})
                break
            except:
                time.sleep(60)
        responses.append(response.content)

    summary = ''
    for i, response in enumerate(responses):
        summary += f'CHUNK {i}:\n{response}\n\n'
    break

 80%|███████▉  | 185/232 [00:00<00:00, 500287.71it/s]


In [ ]:
path_syn = '../dump/synthesis'
path_html = '../dump/htmls'

i = 0
doi = dois[i]
with open(os.path.join(path_syn, doi.replace('/','%') + '.json')) as f:
    output_data = json.load(f)
xs = []
for data in output_data:
    for k, v in data.items():
        if k == 'etc': continue
        if isinstance(v, list):
            for x in v:
                xs.append(x)
        elif isinstance(v, str):
            xs.append(v)
        elif isinstance(v, (int, float)):
            xs.append(str(v))

chunk_size = 140
for (_, header), section in docs[doi].items():
    print('='*chunk_size)
    print(f'SECTION: {header}'.center(chunk_size))
    print('='*chunk_size)
    for (_, p) in section:
        sentences = to_chunk(p, chunk=chunk_size)
        sentences = highlight_text(sentences, xs)
        print_chunk(sentences)

# Transformers

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from typing import List, Dict
import os

class TextBasedLLM:
    def __init__(self, model_path: str, device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        """
        Args:
            model_path: 로컬에 저장된 모델 경로 (예: "meta-llama/Llama-3.2-1B")
            device: 사용할 디바이스 ("cuda" 또는 "cpu")
        """
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            low_cpu_mem_usage=True
        ).to(device)
        
        self.conversation_history: List[Dict] = []
                    
    def generate_response(self, prompt: str, max_length: int = 2048) -> str:
        """응답 생성"""
        # 대화 기록을 포함한 전체 프롬프트 생성
        full_prompt = self._build_prompt(prompt)
        
        # 입력 인코딩
        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(self.device)
        
        # 응답 생성
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=max_length,
                num_return_sequences=1,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # 프롬프트 부분 제거하여 실제 응답만 추출
        response = response[len(full_prompt):].strip()
        
        # 대화 기록 업데이트
        self.conversation_history.append({"role": "user", "content": prompt})
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return response
    
    def _build_prompt(self, prompt: str) -> str:
        """대화 기록을 포함한 프롬프트 생성"""
        conversation = ""
        for message in self.conversation_history[-4:]:  # 최근 4개 메시지만 포함
            conversation += f"{message['role']}: {message['content']}\n"
        return f"{conversation}user: {prompt}\nassistant:"
    
    def start_conversation(self, context: str):
        """대화 시작"""

        # 시스템 프롬프트 설정
        system_prompt = (
            "You are a helpful AI assistant. You will be provided with a text document, "
            "and you should answer questions based on that document. "
            "If the answer cannot be found in the document, say so.\n\n"
            f"Document content:\n{context}\n"
        )
        self.conversation_history = [{"role": "system", "content": system_prompt}]
        
        print("대화를 시작합니다. 종료하려면 'quit' 또는 'exit'를 입력하세요.")
        
        while True:
            user_input = input("\nUser: ").strip()
            
            if user_input.lower() in ['quit', 'exit']:
                print("대화를 종료합니다.")
                break
                
            try:
                response = self.generate_response(user_input)
                print(f"\nAssistant: {response}")
            except Exception as e:
                print(f"Error: {e}")
                continue


class TokenOptimizedChatMemory:
    def __init__(self, model_name="gpt-3.5-turbo", max_memory_tokens=500):
        self.memory = []  # 대화 메모리를 저장하는 리스트
        self.max_memory_tokens = max_memory_tokens
        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.generator = pipeline("text-generation", model=model_name, device=0)
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')  # 임베딩 모델

    def _calculate_memory_tokens(self):
        """현재 메모리의 총 토큰 수 계산."""
        return sum(len(self.tokenizer.encode(item)) for item in self.memory)

    def _truncate_memory(self):
        """토큰 제한 초과 시, 중요도를 고려하여 메모리 최적화."""
        total_tokens = self._calculate_memory_tokens()

        if total_tokens <= self.max_memory_tokens:
            return

        # 임베딩 기반 중요도 계산
        embeddings = self.embedder.encode(self.memory)
        central_embedding = np.mean(embeddings, axis=0).reshape(1, -1)
        scores = cosine_similarity(embeddings, central_embedding).flatten()

        # 중요도가 낮은 순서대로 삭제
        sorted_indices = np.argsort(scores)  # 낮은 중요도 순서대로 정렬
        while total_tokens > self.max_memory_tokens and sorted_indices.size > 0:
            idx_to_remove = sorted_indices[0]
            self.memory.pop(idx_to_remove)
            sorted_indices = np.delete(sorted_indices, 0)  # 제거된 인덱스 업데이트
            total_tokens = self._calculate_memory_tokens()

    def extract_and_store_info(self, user_input):
        """사용자 입력에서 정보를 추출 및 저장."""
        summary_prompt = f"Summarize the key information from this input:\n{user_input}\nSummary:"
        summary = self.generator(summary_prompt, max_length=50, num_return_sequences=1)[0]['generated_text']
        self.memory.append(summary)
        self._truncate_memory()

    def update_memory(self, update):
        """새로운 정보를 추가 및 메모리 최적화."""
        self.memory.append(update)
        self._truncate_memory()

    def generate_response(self, user_input):
        """메모리를 기반으로 GPT-3 응답 생성."""
        memory_context = " ".join(self.memory)
        prompt = f"Memory:\n{memory_context}\nUser input:\n{user_input}\nResponse:"
        response = self.generator(prompt, max_length=150, num_return_sequences=1)[0]['generated_text']
        return response.strip()




In [305]:

MODEL_PATH = "meta-llama/Llama-3.2-1B"

llm = TextBasedLLM(MODEL_PATH)

TEXT_PATH = "path/to/your/text/file.txt"
    
llm.start_conversation()

<module 'llama_stack' from '/home/jhyang/anaconda3/envs/llm/lib/python3.9/site-packages/llama_stack/__init__.py'>

In [120]:
model_path = 'meta-llama/Llama-3.2-3B'
tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
            model_path,
            low_cpu_mem_usage=True,
            output_hidden_states=True,
            pad_token_id=tokenizer.pad_token_id,
            max_length=4096,
        ).to('cuda')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## prompt

In [152]:
prompt = open('../dump/prompt/trans_v1.txt').read()

def extract_information(text, memory=''):
    full_text = prompt
    full_text += f'<START OF MEMORY>\n{memory}\n<END OF MEMORY>' if len(memory) != 0 else ''
    full_text += f'<START OF TEXT>:\n{text}<END OF TEXT>\n\nInformation:\n'

    inp = tokenizer(full_text, return_tensors="pt").to('cuda')
    with torch.no_grad():
        out = model.generate(**inp,
                            repetition_penalty=1.2,  # 패널티 값 설정
                            no_repeat_ngram_size=2,)
    output_text = tokenizer.decode(out[0][0])
    return output_text#.replace(full_text,'')
#    

In [153]:
out = extract_information(doc[(82,'Abstract')][0][1])

/home/jhyang/anaconda3/envs/llm/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:774: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(


In [ ]:
print(out)

In [ ]:
text = doc[(82,'Abstract')][0][1]
for i in range(0, len(text), 80):
    print(text[i:i+80])

# parsing json

In [28]:
from pymatgen.core import Composition

data_path = os.path.join(data_root, 'parsed_lv1')
pattern_comp = re.compile(r"\[([^\]]+)\]")
pattern_val = re.compile(r'(\d+(?:\.\d+)?)')
pattern_unit_c = re.compile(r'([°º]C|℃)')

test_data = []

for fn in os.listdir(data_path):
    with open(os.path.join(data_path, fn)) as f:
        js = json.load(f)
    doi = fn.replace('%','/').replace('.json','')
    for j in js:
        if 'solid-state' not in j['method']:
            continue
        target_chem = j['target']
        precursor_comp = [Composition(prec).as_dict() for prec in j['precursors']]
        heat_time = []
        if j['time'] is None:
            pass
        elif isinstance(j['time'], list):
            heat_time = [pattern_val.findall(t_) for t_ in j['time']]
        elif isinstance(j['time'], str):
            heat_time = [pattern_val.findall(j['time'])]
        if len(heat_time) != 0:
            heat_time = np.hstack(heat_time).astype(float).tolist()

        heat_temp_ = []
        if j['temperature'] is None:
            pass
        elif isinstance(j['temperature'], list):
            heat_temp_ = j['temperature']
        elif isinstance(j['temperature'], str):
            heat_temp_ = [j['temperature']]

        heat_temp = []
        for t in heat_temp_:
            v = float(pattern_val.findall(t)[0])
            if pattern_unit_c.search(t) is None:
                heat_temp.append(v)
            else:
                heat_temp.append(v + 273.15)

        base = {
            'doi':doi,
            'target_comp':None,
            'precursor_comp':precursor_comp,
            'heat_time':heat_time,
            'heat_temp':heat_temp,
        }
        if 'etc' in j.keys():
            base.update({'etc':j['etc']})

        if 'x_values' in j.keys():
            target_chem = re.sub(r'(\d+\.?\d*)(x)', r'\1*\2', target_chem)
            target_chem = re.sub(r'(\.\d+)(x)', r'\1*\2', target_chem)
            for x_val in j['x_values']:
                out = base.copy()
                _target_chem = target_chem
                if x_val is None: continue
                for pattern_x in pattern_comp.findall(target_chem):
                    v = round(float(sympify(pattern_x).subs(x, x_val)), 8)
                    _target_chem = _target_chem.replace(f'[{pattern_x}]',str(v))
                out['target_comp'] = Composition(_target_chem).as_dict()
                test_data.append(out)
        else:
            out = base.copy()
            out['target_comp'] = Composition(target_chem).as_dict()
            test_data.append(out)
        

In [29]:
with open(os.path.join(data_root, 'parsed_lv2.pkl'),'wb') as f:
    pickle.dump(test_data, f)

In [27]:
data_path = os.path.join(data_root, 'parsed_lv1')
pattern_comp = re.compile(r"\[([^\]]+)\]")
pattern_val = re.compile(r'(\d+(?:\.\d+)?)')
pattern_unit_c = re.compile(r'([°º]C|℃)')

x = symbols('x')
fn = '10.1039%D1DT01780B.json'
with open(os.path.join(data_path, fn)) as f:
    js = json.load(f)
doi = fn.replace('%','/').replace('.json','')
for j in js:
    if 'solid-state' not in j['method']:
        continue
    target_chem = j['target']
#    precursor_comp = [Composition(prec).as_dict() for prec in j['precursors']]
    heat_time = []
    if j['time'] is None:
        pass
    elif isinstance(j['time'], list):
        heat_time = [pattern_val.findall(t_) for t_ in j['time']]
    elif isinstance(j['time'], str):
        heat_time = [pattern_val.findall(j['time'])]
    if len(heat_time) != 0:
        heat_time = np.hstack(heat_time).astype(float).tolist()

    heat_temp_ = []
    if j['temperature'] is None:
        pass
    elif isinstance(j['temperature'], list):
        heat_temp_ = j['temperature']
    elif isinstance(j['temperature'], str):
        heat_temp_ = [j['temperature']]

    heat_temp = []
    for t in heat_temp_:
        v = float(pattern_val.findall(t)[0])
        if pattern_unit_c.search(t) is None:
            heat_temp.append(v)
        else:
            heat_temp.append(v + 273.15)

    base = {
        'doi':doi,
        'target_comp':None,
#        'precursor_comp':precursor_comp,
        'heat_time':heat_time,
        'heat_temp':heat_temp,
    }
    if 'etc' in j.keys():
        base.update({'etc':j['etc']})

    if 'x_values' in j.keys():
        target_chem = re.sub(r'(\d+\.?\d*)(x)', r'\1*\2', target_chem)
        target_chem = re.sub(r'(\.\d+)(x)', r'\1*\2', target_chem)
        for x_val in j['x_values']:
            out = base.copy()
            _target_chem = target_chem
            if x_val is None: continue
            for pattern_x in pattern_comp.findall(target_chem):
                v = round(float(sympify(pattern_x).subs(x, x_val)), 8)
                _target_chem = _target_chem.replace(f'[{pattern_x}]',str(v))
                print(_target_chem, x_val, pattern_x, v)
#            out['target_comp'] = Composition(target_chem).as_dict()
#            test_data.append(out)
            print(_target_chem)


Ba2Gd1.9Eu[x]Ge4O13 0.1 2-x 1.9
Ba2Gd1.9Eu0.1Ge4O13 0.1 x 0.1
Ba2Gd1.9Eu0.1Ge4O13
Ba2Gd1.6Eu[x]Ge4O13 0.4 2-x 1.6
Ba2Gd1.6Eu0.4Ge4O13 0.4 x 0.4
Ba2Gd1.6Eu0.4Ge4O13
Ba2Gd1.4Eu[x]Ge4O13 0.6 2-x 1.4
Ba2Gd1.4Eu0.6Ge4O13 0.6 x 0.6
Ba2Gd1.4Eu0.6Ge4O13
Ba2Gd1.2Eu[x]Ge4O13 0.8 2-x 1.2
Ba2Gd1.2Eu0.8Ge4O13 0.8 x 0.8
Ba2Gd1.2Eu0.8Ge4O13
